# Fine-Tuning Qwen2.5-0.5B-Instruct with LoRA

This notebook fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` on the Alpaca-cleaned dataset using LoRA (via PEFT + TRL), then merges the adapter back into the base model and exports it as a zip archive.

**Flow:**
1. Setup & Installs
2. Configuration
3. Load Tokenizer & Base Model
4. Prepare Dataset
5. Configure LoRA
6. Train
7. Save Adapter
8. Merge LoRA into Base Model
9. Export

## 1. Setup & Installs

In [5]:
!pip uninstall -y transformers peft trl accelerate datasets torchao

!pip install -q \
    transformers==4.53.3 \
    peft==0.15.2 \
    trl==0.18.2 \
    accelerate==1.8.1 \
    datasets==4.0.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 67.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.4/366.4 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.5 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [6]:
import os
import shutil

import torch
import pandas as pd

import transformers
import peft
import trl

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer

print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)
print("trl         :", trl.__version__)


transformers: 4.53.3
peft        : 0.15.2
trl         : 0.18.2


In [7]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA Available: True
GPU: Tesla T4


## 2. Configuration

All tunable settings live here so the rest of the notebook never hardcodes them again.

In [13]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

DATA_FILE = "/kaggle/input/datasets/thedevastator/alpaca-language-instruction-training/train.csv"
NUM_SAMPLES = 5000
MAX_LENGTH = 512
TEST_SIZE = 0.1
SEED = 42

ADAPTER_DIR = "qwen_lora_adapter"
MERGED_DIR = "qwen_merged_model"
OUTPUT_DIR = "./qwen-lora"


## 3. Load Tokenizer & Base Model

Loaded once here and reused everywhere below (no repeated loading).

In [14]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


In [15]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2RotaryEmbe

## 4. Prepare Dataset

In [16]:
dataset = load_dataset("csv", data_files=DATA_FILE)
dataset = dataset["train"].select(range(NUM_SAMPLES))

print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'output', 'input'],
    num_rows: 5000
})


Format each example into an instruction/response prompt template.

In [17]:
def format_prompt(example):
    instruction = str(example["instruction"]).strip()
    output = str(example["output"]).strip()

    if pd.isna(example["input"]) or str(example["input"]).strip() == "":
        text = f"""### Instruction:
{instruction}

### Response:
{output}"""
    else:
        input_text = str(example["input"]).strip()
        text = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{output}"""

    return {"text": text}


dataset = dataset.map(format_prompt)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [18]:
dataset = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Training samples  :", len(train_dataset))
print("Validation samples:", len(eval_dataset))


Training samples  : 4500
Validation samples: 500


Tokenize and drop the now-unneeded raw text columns.

In [19]:
def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


train_dataset = train_dataset.map(tokenize_function)
eval_dataset = eval_dataset.map(tokenize_function)

train_dataset = train_dataset.remove_columns(["instruction", "input", "output", "text"])
eval_dataset = eval_dataset.remove_columns(["instruction", "input", "output", "text"])


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## 5. Configure LoRA

In [20]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


## 6. Train

In [21]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

hf_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [22]:
hf_trainer.train()


Epoch,Training Loss,Validation Loss
1,1.403900,1.403732
2,1.316300,1.403319


TrainOutput(global_step=1126, training_loss=1.3507222697324057, metrics={'train_runtime': 1173.4402, 'train_samples_per_second': 7.67, 'train_steps_per_second': 0.96, 'total_flos': 9954960998400000.0, 'train_loss': 1.3507222697324057, 'epoch': 2.0})

## 7. Save Adapter

In [23]:
hf_trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)


('qwen_lora_adapter/tokenizer_config.json',
 'qwen_lora_adapter/special_tokens_map.json',
 'qwen_lora_adapter/chat_template.jinja',
 'qwen_lora_adapter/vocab.json',
 'qwen_lora_adapter/merges.txt',
 'qwen_lora_adapter/added_tokens.json',
 'qwen_lora_adapter/tokenizer.json')

## 8. Merge LoRA into Base Model

Reloads a fresh copy of the base model (without `device_map`, so it can be merged on CPU/GPU as a single unit) and merges the trained adapter into it.

In [24]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()


In [25]:
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)


('qwen_merged_model/tokenizer_config.json',
 'qwen_merged_model/special_tokens_map.json',
 'qwen_merged_model/chat_template.jinja',
 'qwen_merged_model/vocab.json',
 'qwen_merged_model/merges.txt',
 'qwen_merged_model/added_tokens.json',
 'qwen_merged_model/tokenizer.json')

## 9. Export

In [26]:
print(os.listdir("/kaggle/working"))


['.virtual_documents', 'qwen-lora', 'qwen_merged_model', 'qwen_lora_adapter']


In [27]:
shutil.make_archive(
    f"/kaggle/working/{MERGED_DIR}",
    "zip",
    f"/kaggle/working/{MERGED_DIR}",
)

print("Zip file created!")


Zip file created!
